# DeltaNet v2: Performance Benchmark on Google Colab (Tesla T4)
### Comparing DeltaNet v2 (SM75 CuTe, C++17) against NVIDIA GatedDeltaNet-2 and FLA

This notebook runs the comprehensive benchmark suite from `/content/Deltanet/tests/` to evaluate:
- **Runtime Performance**: Latency (ms) and Throughput (tokens/sec)
- **Peak Memory Consumption**: VRAM allocated & reserved (MB)

**Baselines Compared:**
1. **DeltaNet v2 (Ours)**: Fused CUDA kernels targeting NVIDIA Turing SM75 (Tesla T4, RTX 20-series) with independent $k, v$ gating, compiled with C++17.
2. **FLA GatedDeltaNet-2**: NVIDIA's official Gated DeltaNet-2 paper architecture implemented via Flash Linear Attention (`fla.layers.GatedDeltaNet2`).
3. **FLA GatedDeltaNet (v1)**: Standard Gated DeltaNet linear attention layer (`fla.layers.GatedDeltaNet`).
4. **PyTorch Reference**: Unfused PyTorch eager baseline.

## 1. Verify GPU Hardware
Confirm that Google Colab is running with an **NVIDIA Tesla T4** GPU (SM75, 16 GB VRAM).

In [ ]:
!nvidia-smi

## 2. Downgrade to Python 3.10 (C++17 & PyTorch 2.4 Environment)
> **Why downgrade?**
> Google Colab's default environment uses Python 3.13 with PyTorch 2.6+, which enforces C++20 (`#error C++20 or later compatible compiler is required to use PyTorch.`) and breaks compatibility with many custom CUDA extensions.
> Downgrading to **Python 3.10** with **PyTorch 2.4.1 (cu121)** provides a battle-tested, stable environment using standard **C++17**.

In [ ]:
# 1. Install Python 3.10 and development headers
!sudo apt-get update -q
!sudo apt-get install -y software-properties-common
!sudo add-apt-repository -y ppa:deadsnakes/ppa || true
!sudo apt-get update -q
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils libpython3.10-dev

# 2. Configure symlinks so python and pip point to Python 3.10
!sudo ln -sf /usr/bin/python3.10 /usr/local/bin/python
!sudo ln -sf /usr/bin/python3.10 /usr/local/bin/python3
!sudo ln -sf /usr/bin/python3.10 /usr/bin/python
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10
!sudo ln -sf /usr/local/bin/pip /usr/bin/pip || true

# 3. Verify Python version
!python --version

# 4. Install PyTorch 2.4.1 with CUDA 12.1 and build essentials (C++17 compatible)
!python -m pip install -q --upgrade pip setuptools wheel ninja einops
!python -m pip install -q torch==2.4.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 5. Sanity check PyTorch CUDA availability
!python -c "import torch; print(f'PyTorch {torch.__version__} | CUDA Available: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"None\"}')"

## 3. Clone Repository & Setup DeltaNet v2
Clones `https://github.com/CBG6682/Varlen-Gated-DeltaNet.git` into `/content/Deltanet`.

You can choose between **two options**:
- **Option 3A (Recommended - Instant Setup in ~5s)**: Uses the pre-compiled CUDA SM75 extension included directly in the repo for Tesla T4 / Python 3.10. Ready to run immediately with zero compilation wait time!
- **Option 3B (Developer Mode - Recompile from source)**: Recompiles CUDA kernels using `nvcc` and C++17 (run this if you modified code in `src/` or `include/`).

In [ ]:
# ==============================================================================
# Option 3A (Recommended): Instant Setup using Pre-compiled CUDA Extension (~5s)
# ==============================================================================
!rm -rf /content/Deltanet
!git clone https://github.com/CBG6682/Varlen-Gated-DeltaNet.git /content/Deltanet
%cd /content/Deltanet

# Install package into Python 3.10 environment using pre-compiled extension
!USE_PREBUILT=1 python -m pip install -q -e . --no-build-isolation

### (Optional) Option 3B: Recompile CUDA Kernels from Source
*Run this cell only if you modified the CUDA/C++ source code or wish to re-generate the binary.*

In [ ]:
# ==============================================================================
# Option 3B (Developer Mode): Recompile from source with C++17
# ==============================================================================
%cd /content/Deltanet
!git submodule update --init --depth 1 third_party/cutlass || true

# Limit parallel compilation jobs to 2 to prevent memory exhaustion on Colab
%env MAX_JOBS=2

# Compile and install DeltaNet v2 from source
!MAX_JOBS=2 python -m pip install -v . --no-build-isolation
!cp build/lib.*/deltanet_v2/_C*.so deltanet_v2/ 2>/dev/null || true

## 4. Install Baseline Libraries
Installs **Flash Linear Attention** (`fla`) and **NVIDIA GatedDeltaNet-2** (`NVlabs/GatedDeltaNet-2`).

In [ ]:
# Install Flash Linear Attention (FLA)
!python -m pip install -q flash-linear-attention

# Install NVlabs GatedDeltaNet-2
!python -m pip install -q git+https://github.com/nvlabs/gateddeltanet-2.git --no-deps || true

## 5. Numerical Correctness Sanity Test
Verifies numerical consistency between DeltaNet v2 CUDA kernels and the pure PyTorch reference.

In [ ]:
!python /content/Deltanet/tests/test_correctness.py

## 6. Benchmark: Autoregressive Inference (L = 1)
Measures single-step decode latency (ms/step), generation throughput (tokens/sec), and peak VRAM across batch sizes $B \in [1, 16, 64, 256, 1024, 2048]$.

In [ ]:
!python /content/Deltanet/tests/bench_inference.py --dim 768 --q-heads 8 --kv-heads 8 --batches 1,16,64,256,1024,2048

## 7. Benchmark: Training Pass (Forward + Backward)
Measures step time (ms), training throughput (tokens/sec), and peak VRAM across sequence lengths $L \in [512, 1024, 2048, 4096]$.

In [ ]:
!python /content/Deltanet/tests/bench_training.py --dim 768 --q-heads 8 --kv-heads 8 --batch-size 4 --seq-lens 512,1024,2048,4096

## 8. Run Full Benchmark Suite
Runs all benchmarks and prints consolidated comparison tables.

In [ ]:
!python /content/Deltanet/tests/bench_all.py